In [ ]:
# Copyright (c) 2025 by Miguel A. Caro, Aalto University (miguel.caro@aalto.fi, mcaroba@gmail.com)

##################################################################################################
#
# This code will perform simulated experiments of generation of graphitic (amorphous) carbons
# with a graphitization (high temperature annealing) + quenching process. The user can select the
# graphitization time tg with index ig = [1, 2, 3, 4, 5, 6], where tg = 10**(ig/2+0.5) ps, the
# quenching time tq with index iq = [1, 2, 3, 4, 5, 6, 7], where tg = 10**(iq/2-0.5) ps, and the
# number of experiments (up to five) for each (ig, iq) combination. Each ns of simulation costs
# 1 Alvar that needs to be paid out of the user's budget. Upon collecting data, the code will
# train a machine learning model that will predict the product of the Bulk modulus and the
# specific volume (inverse density) of the material. The objective is to find a recipe to make
# a material with both low density (high specific volume) and high Bulk modulus, noting that in
# carbon materials the trend is that the higher the density the higher the Bulk modulus. We
# want to find an optimum. The "game" dynamics is to leverage the predictions of the ML model
# (including the predicted model uncertainty) and balance exploration vs exploitation of the
# prediction to minimize the cost (~ number of experiments) of finding the optimum.
#
# The "game" is complicated by the fact that experiments with low graphitization and/or quenching
# times are more expensive to run (1 Alvar/ns, rounding up!) and that individual data points can
# be quite noisy. I.e., to get a reliable measurement of B and v at a given (ig, iq) combination,
# it is best to "purchase" all five data points available. The ML model implicitly takes
# into account noise, by using regularization, when several measurements are available at the same
# (ig, iq) combination (we could even estimate the regularization parameter based on the standard
# deviation). The pedagogical value of the game is to make the student think in terms of
# optimizing the value of the gathered data towards achieving the goal of optimizing the desired
# material properties. This type of task can be further automated by using an
# "acquision function" to decide which experiments to performed based on the expected benefit.
# This type of problem belongs to Bayesian optimization and, more generally, is an example of
# reinforcement learning.
#
# You should only run this top block of code once, to initialize all the functions and variables.
# You should rerun the bottom block of code every time you want to add more data to the model.
#
##################################################################################################

# Module imports
import numpy as np
import ipywidgets as widgets
from IPython.display import display
import matplotlib as mpl
import matplotlib.pyplot as plt
from time import sleep

# Preload all the available "experiments" to avoid having to run them on the fly. Measurements are extracted from here
exps = np.loadtxt("data.dat")
data0 = {}
for exp in exps:
    ig, iq, r = exp[0], exp[1], exp[2] # graphitization (1-6), quenching (1-7) and random (1-5) indices, respectively
                                       # for graph, the indices correspond to log10(graphitization time [in ps]) = 1,
                                       # 1.5, 2., 2.5, 3, 3.5; for quenching, log10(graphitization time [in ps]) = 0,
                                       # 0.5, 1., 1.5, 2., 2.5, 3.; random index just refers to independent seeds
    if (ig, iq) not in data0:
        data0[(ig, iq)] = []
        data0[(ig, iq)].append({})
        data0[(ig, iq)][-1]["seed"] = r # random seed for this experiment
        data0[(ig, iq)][-1]["B"] = exp[3] # Bulk modulus for this experiment
        data0[(ig, iq)][-1]["v"] = 1./exp[4] # Specific volume (inverse density) for this experiment
    else:
        data0[(ig, iq)].append({})
        data0[(ig, iq)][-1]["seed"] = r # random seed for this experiment
        data0[(ig, iq)][-1]["B"] = exp[3] # Bulk modulus for this experiment
        data0[(ig, iq)][-1]["v"] = 1./exp[4] # Specific volume (inverse density) for this experiment

# This function plots a list of images side by side
def display_imgs_sidebyside(img_list, E_list, v_list):
    imgs = []
    for img_path in img_list:
        imgs.append(open(img_path, 'rb').read())
    hor = []
    for i in range(0, len(imgs)):
        vert = [widgets.Image(value=imgs[i], format='png', width=210, height=210),
                widgets.Button(description="B = %4.0f GPa; v = %4.2f cm^3/g" % (E_list[i], v_list[i]),
                               layout=widgets.Layout(width="210px", height='40px'))]
        hor.append(widgets.VBox(vert))
    display(widgets.HBox(hor))

# We need to define a Budget class to reduce the budget with each measurement
class Budget:
    def __init__(self, initial_budget):
        self.remaining = initial_budget

def decrease_budget(budget, n):
    budget.remaining -= n

# This function performs measurements: essentially, it moves entries from the "hidden" data0 to the "visible" data
def measure(data0, data, ig, iq, n_exp, budget):
    # We perform n_exp measurements
    img_list = []
    B_list = []
    v_list = []
    for i in range(0, n_exp):
        cost = np.ceil( (10.**(ig/2. + 0.5) + 10.**(iq/2. - 0.5)) / 1000. ) # 1 Alvar for each ns of simulation
        if cost <= budget.remaining and len(data0[(ig,iq)]) > 0: # Check we have enough Alvars and experiments left
            i_rand = np.random.choice(len(data0[(ig,iq)]))
            this_data = data0[(ig,iq)][i_rand]
            this_data["ig"] = ig
            this_data["iq"] = iq
            del data0[(ig,iq)][i_rand]
            data.append(this_data)
            budget.remaining -= cost
            print("Experiment %i/%i successful! Remaining budget: %.0f" % (i+1, n_exp, budget.remaining))
            img_list.append("%i/%i_%i.png" % (this_data["seed"], ig, iq))
            B_list.append(this_data["B"])
            v_list.append(this_data["v"])
        elif len(data0[(ig,iq)]) == 0:
            print("No more experiments available for the given (ig,iq): nothing to do!")
        elif cost > budget.remaining:
            print("Your remaining budget of %.0f is not enough to carry out requested experiment %i/%i" % (budget.remaining, i+1, n_exp))
    if len(img_list) > 0:
        display_imgs_sidebyside(img_list, B_list, v_list)

# Our ML model architecture is based on Gaussian process regression. We will use it to regress B, v and B*v
# Our Bayesian optimization procedure will be driven by maximizing B*v. We could also use other objective
# functions, e.g., B*v**2 (to emphasize larger v) but note that the prior variances would need to be updated
# We need to define our priors, which we can base on intuitive values. The noise levels can in principle be
# updated once we have more than one data point per (ig, iq) combination by setting a per-data-point
# regularization parameter but we're not going to do it here
B0 = 20. # Expected Bulk modulus in GPa
v0 = 1./1.25 # Expected specific volume in cm^3/g
Bv0 = B0*v0
sigma_B = 0.1*B0 # Expected noise is set to 10% of the expected value
sigma_v = 0.1*v0 # Expected noise is set to 10% of the expected value
sigma_Bv = np.sqrt(B0**2*sigma_v**2 + v0**2*sigma_B**2) # Rough estimate of the noise in the product B*v
delta_B = 5. # Expected variance (range of values for this quantity)
delta_v = 0.25 # Expected variance (range of values for this quantity)
delta_Bv = np.sqrt(B0**2*delta_v**2 + v0**2*delta_B**2) # Rough estimate
delta_ij = 2. # To define the covariance between grid points in (ig, iq), noting we use a logarithmic grid
               # We could give different sigmas for ig and iq, but here the dimensions are the same

# Define the covariance function
def cov(i1, j1, i2, j2, delta_Bv, delta_ij):
    return delta_Bv**2 * np.exp(-0.5*( (i1-i2)**2 + (j1-j2)**2 )/delta_ij**2 )

# This is where the magic happens: here we train our model and evaluate it in the entire grid
def train_and_evaluate(data, Bv0, delta_Bv, sigma_Bv, delta_ij):
    # Measured objective function values
    Bv = np.zeros(len(data))
    for i in range(0, len(data)):
        Bv[i] = data[i]["B"] * data[i]["v"]
    # Compute grid properties (where we evaluate the model)
    cov_m = np.zeros( [6*7, 6*7] )
    for k1 in range(0, 6*7):
        i1 = k1 % 6
        j1 = k1 // 6
        cov_m[k1, k1] = delta_Bv**2
        for k2 in range(k1+1, 6*7):
            i2 = k2 % 6
            j2 = k2 // 6
            cov_m[k1, k2] = cov(i1, j1, i2, j2, delta_Bv, delta_ij)
            cov_m[k2, k1] = cov_m[k1, k2]
    # Compute data properties
    sigma = np.zeros( len(data) ) + sigma_Bv # Default regularization for all points
    cov_m_data = np.zeros( [len(data), len(data)] )
    for k1 in range(0, len(data)):
        i1 = data[k1]["ig"]
        j1 = data[k1]["iq"]
        cov_m_data[k1, k1] = delta_Bv**2 + sigma[k1]**2 # Gaussian noise regularization
        for k2 in range(k1+1, len(data)):
            i2 = data[k2]["ig"]
            j2 = data[k2]["iq"]
            cov_m_data[k1, k2] = cov(i1, j1, i2, j2, delta_Bv, delta_ij)
            cov_m_data[k2, k1] = cov_m_data[k1, k2]
    # Invert the covariance matrix of the observed data
    cov_m_data_inv = np.linalg.pinv(cov_m_data)
    # Compute covariance between training and evaluation points
    cov_m_data2 = np.zeros([6*7, len(data)])
    alphas = np.dot(cov_m_data_inv, Bv-Bv0) # Fitting coefficients (or "weights")
    mu_model = np.zeros(6*7)
    for k1 in range(0, 6*7):
        i1 = k1 % 6 + 1
        j1 = k1 // 6 + 1
        covs = np.zeros(len(data))
        for k2 in range(0, len(data)):
            i2 = data[k2]["ig"]
            j2 = data[k2]["iq"]
            covs[k2] = cov(i1, j1, i2, j2, delta_Bv, delta_ij)
        cov_m_data2[k1,:] = covs
        # Best model at the evaluation points
        mu_model[k1] = Bv0 + np.dot(alphas, covs)
    # The diagonal of this is the expected variance at the evaluation points (aka, the uncertainty of the model)
    cov_m_model = cov_m - np.dot(np.dot(cov_m_data2, cov_m_data_inv), np.transpose(cov_m_data2))
    # Plotting
    #
    # Plot model's current predictions and uncertainty
    x = np.zeros([6,7]); y = np.zeros([6,7])
    z = np.zeros([6,7]); z_err = np.zeros([6,7])
    # Put values on grid
    for j in range(0, 7):
        for i in range(0, 6):
            k = i + j*6
            x[i, j] = i+1
            y[i, j] = j+1
            z[i, j] = mu_model[k]
            z_err[i, j] = np.sqrt(cov_m_model[k,k])
    # Matplotlib magic
    fig, axs = plt.subplots(1, 2, figsize=(15,5))
    cmaplist = [(0.5,0,1,1), (0,0,1,1), (0,1,0,1), (1,1,0,1), (1,0,0,1)]
    cmap = mpl.colors.LinearSegmentedColormap.from_list('Custom cmap', cmaplist, 20)
    # Model
    axs[0].set_aspect('equal', 'box')
    axs[0].set(xlim=(0,7), ylim=(0,8))
    axs[0].set_title("Model")
    axs[0].set_xlabel("Graphitization time (log scale: 1 = 10 ps; 5 = 1 ns)")
    axs[0].set_ylabel("Quenching time (log scale: 1 = 1 ps; 7 = 1 ns)")
    axs[0].set_xticks(np.arange(1,7,1))
#    axs[0].set_xticklabels(["10 ps", "32 ps", "100 ps", "316 ps", "1 ns", "3.2 ns"])
    axs[0].set_yticks(np.arange(1,8,1))
    psm = axs[0].imshow(np.transpose(z), vmin=0, vmax=40, rasterized=True, cmap=cmap,
                        interpolation="none", origin="lower", extent=(0.5,6.5,0.5,7.5))
    fig.colorbar(psm, ax=axs[0], ticks=[0,5,10,15,20,25,30,35,40], label="Model prediction of B/rho (GPa*cm^3/g)")
    axs[0].plot(x.flatten(), y.flatten(), "+")
    x_dat = [data[i]["ig"] for i in range(0, len(data))]
    y_dat = [data[i]["iq"] for i in range(0, len(data))]
    axs[0].plot(x_dat, y_dat, "x", markersize=8, markeredgewidth=3, color="black")
    axs[0].plot(x_dat, y_dat, "x", markersize=6, markeredgewidth=1, color="red")
    #x_dat = [data[i][0] for i in range(0, len(data)) if data[i][3] == "sim"]
    #y_dat = [data[i][1] for i in range(0, len(data)) if data[i][3] == "sim"]
    #axs[0].plot(x_dat, y_dat, "x", markersize=8, markeredgewidth=3, color="black")
    #axs[0].plot(x_dat, y_dat, "x", markersize=6, markeredgewidth=1, color="green")
    # Model uncertainty
    axs[1].set_aspect('equal', 'box')
    axs[1].set(xlim=(0,7), ylim=(0,8))
    axs[1].set_title("Model uncertainty")
    axs[1].set_xlabel("Graphitization time (log scale: 1 = 10 ps; 5 = 1 ns)")
    axs[1].set_ylabel("Quenching time (log scale: 1 = 1 ps; 7 = 1 ns)")
    axs[1].set_xticks(np.arange(1,7,1))
    axs[1].set_yticks(np.arange(1,8,1))
    psm = axs[1].imshow(np.transpose(z_err), vmin=0, vmax=5., rasterized=True, cmap=cmap,
                        interpolation="none", origin="lower", extent=(0.5,6.5,0.5,7.5))
    fig.colorbar(psm, ax=axs[1], ticks=[0,1,2,3,4,5], label="Model uncertainty (GPa*cm^3/g)")
    axs[1].plot(x.flatten(), y.flatten(), "+")
    x_dat = [data[i]["ig"] for i in range(0, len(data))]
    y_dat = [data[i]["iq"] for i in range(0, len(data))]
    axs[1].plot(x_dat, y_dat, "x", markersize=8, markeredgewidth=3, color="black")
    axs[1].plot(x_dat, y_dat, "x", markersize=6, markeredgewidth=1, color="red")
    #x_dat = [data[i][0] for i in range(0, len(data)) if data[i][3] == "sim"]
    #y_dat = [data[i][1] for i in range(0, len(data)) if data[i][3] == "sim"]
    #axs[1].plot(x_dat, y_dat, "x", markersize=8, markeredgewidth=3, color="black")
    #axs[1].plot(x_dat, y_dat, "x", markersize=6, markeredgewidth=1, color="green")
    #
    plt.show()
    return z, z_err
############################################################################


##################################################################################################
# Initialize our database and our budget
budget = Budget(40.)
data = [{'seed': 1.0, 'B': 10., 'v': 1, 'ig': -100, 'iq': -100}] # Fake data point outside grid to plot priors
z, z_err = train_and_evaluate(data, Bv0, delta_Bv, sigma_Bv, delta_ij)
##################################################################################################

In [ ]:
##################################################################################################
# Run your experiments here!
#
# Remember, for the graphitization and quenching times:
# 
# ig       tg          iq       tq
# -----------          -----------
#  1    10 ps           1     1 ps
#  2    32 ps           2   3.2 ps
#  3   100 ps           3    10 ps
#  4   316 ps           4    32 ps
#  5     1 ns           5   100 ps
#  6  3.16 ns           6   316 ps
#                       7     1 ns
#
# For n_exp you can choose up to 5 per (ig, iq) combination (it doesn't need to be all at the same time)


# Just run this cell changing these options until you run out of Alvars
ig, iq, n_exp = (,,)
measure(data0, data, ig, iq, n_exp, budget = budget)
z, z_err = train_and_evaluate(data, Bv0, delta_Bv, sigma_Bv, delta_ij)

In [ ]:
# We can define an acquisition function to optimize the process. Since we want to
# maximize the quantity z = B/rho, our acquisition function is directly proportional
# to z for the "exploitation" part. We also want to evaluate regions where the model
# error is high, thus the acquisition function is also proportional to z_err. The
# parameter beta controls the tradeoff between exploitation and exploration:
#
# a = z + beta*z_err

# This function finds the location of a maximum for an N-dimensional array (in this example, use with N = 2)
def argmaxN(A, N):
    s = A.shape
    new_shp = s[:-N] + (np.prod(s[-N:]),)
    max_idx = A.reshape(new_shp).argmax(-1)
    return np.unravel_index(max_idx, s[-N:])

# First, reinitialize the data by running the first cell again!!!

# Choose the tradoff between exploration and exploitation
beta = 
# Choose the number of "experiments" per grid point
n_exp = 
for i in range(0,10): # how many times do we run the Bayesian optimization algorithm?
    if budget.remaining == 0:
        print("You ran out of Alvars!")
        break
    a = z + beta * z_err
    ig, iq = argmaxN(a, 2)
    measure(data0, data, ig = ig+1, iq = iq+1, n_exp = n_exp, budget = budget) # we need to add 1 to ig, iq because argmaxN returns 0-based indices
    z, z_err = train_and_evaluate(data, Bv0, delta_Bv, sigma_Bv, delta_ij)
    max_i, max_j = argmaxN(z, 2)
    print("Current maximum predicted at (%i, %i)" % (max_i+1, max_j+1))

In [ ]:
##################################################################################################
# Check a model with all the data to see what it looks like (reload the first code block before)
budget = Budget(1.e10) # crazy high budget
data = []
for i in range(1,7):
    for j in range(1,8):
        measure(data0, data, ig = i, iq = j, n_exp = 5, budget = budget)
z, z_err = train_and_evaluate(data, Bv0, delta_Bv, sigma_Bv, delta_ij)
max_i, max_j = argmaxN(z, 2)
print("Current maximum predicted at (%i, %i)" % (max_i+1, max_j+1))
##################################################################################################